**Set environment**

In [ ]:
suppressMessages(suppressWarnings(source("../run_config_project_sing.R")))
show_env()

## Import data

**Read metadata from reference file**

In [ ]:
### set file path
txt_fdiry = file.path(FD_REF, "encode_chipseq_latest")
txt_fname = "metadata_default_files_240701.tsv"
txt_fpath = file.path(txt_fdiry, txt_fname)

### read table
dat = read_tsv(txt_fpath, show_col_types = FALSE)

### show and assign
dat_metadata_import = dat
print(dim(dat))
fun_display_table(head(dat, 1))

In [ ]:
### set file path
txt_fdiry = file.path(FD_REF, "encode_chipseq_histone")
txt_fname = "metadata.tsv"
txt_fpath = file.path(txt_fdiry, txt_fname)

### read table
dat = read_tsv(txt_fpath, show_col_types = FALSE)

### show and assign
dat_metadata_check = dat
print(dim(dat))
fun_display_table(head(dat, 1))

**Check: Genome**

In [ ]:
dat = dat_metadata_import
table(dat$`File assembly`)

**Check: Assay**

In [ ]:
dat = dat_metadata_import
table(dat$Assay)

**Check: Biosample**

In [ ]:
dat = dat_metadata_import
table(dat$`Biosample term name`)

**Check: File format**

In [ ]:
dat = dat_metadata_import
table(dat$`File format`)

In [ ]:
dat = dat_metadata_import
table(dat$`File format`, dat$`Output type`)

## Get Histone ChIP-seq files

In [ ]:
### init
txt_genome    = "GRCh38"
txt_assay     = "Histone ChIP-seq"
vec_txt_ftype = c("bed narrowPeak", "bigWig")

### filter and arrange
dat = dat_metadata_import
dat = dat %>% 
    dplyr::mutate(
        Biosample        = `Biosample term name`,
        Index_Experiment = `Experiment accession`,
        Index_File       = `File accession`,
        File_Type        = `File format`,
        Output_Type      = `Output type`,
        Genome           = `File assembly`,
        Target           = str_remove(`Experiment target`, "-human"),
        FName            = basename(`File download URL`),
        FUrl             = `File download URL`,
        Analysis         = `File analysis title`
    )  %>%
    dplyr::select(
        Assay, 
        Index_Experiment, 
        Index_File,
        File_Type,
        Output_Type,
        Genome,
        Target,
        Analysis,
        md5sum,
        FName,
        FUrl
    ) %>% dplyr::filter(
        Genome    == txt_genome,
        Assay     == txt_assay,
        File_Type %in% vec_txt_ftype
    )

### assign and show
dat_metadata_histone = dat
print(dim(dat))
fun_display_table(head(dat, 3))

**Check**

In [ ]:
dat  = dat_metadata_histone
vec  = dat$Index_File
vec1 = sort(vec)

dat  = dat_metadata_check
vec  = dat$`File accession`
vec2 = sort(vec)

all(vec1 == vec2)

In [ ]:
dat  = dat_metadata_histone
table(dat$Index_Experiment, dat$File_Type)

## Prepare download file

In [ ]:
### init
dat = dat_metadata_histone

### select file urls
dat = dat %>% dplyr::filter(File_Type == "bed narrowPeak") %>% dplyr::select(FUrl) 

### assign and show
dat_download_file = dat
print(dim(dat))
fun_display_table(head(dat, 3))

## Prepare checksum file

In [ ]:
### init
dat = dat_metadata_histone

### select md5sum for each file
dat = dat %>% dplyr::filter(File_Type == "bed narrowPeak") %>% dplyr::select(md5sum, FName)

### assing and show
dat_download_md5sum = dat
print(dim(dat))
fun_display_table(head(dat))

## Save results

In [ ]:
### set directory
txt_folder = "encode_chipseq_histone"

txt_fdiry = file.path(FD_DAT, "external", txt_folder)
txt_cmd   = paste("mkdir -p", txt_fdiry)
system(txt_cmd)
print(txt_fdiry)

txt_fdiry = file.path(FD_RES, "region", txt_folder, "summary")
txt_cmd   = paste("mkdir -p", txt_fdiry)
system(txt_cmd)
print(txt_fdiry)

In [ ]:
### set directory
txt_fdiry = file.path(FD_DAT, "external", txt_folder)
txt_fname = "files.txt"
txt_fpath = file.path(txt_fdiry, txt_fname)

### write table
dat = dat_download_file
write_tsv(dat, txt_fpath, col_names = FALSE)  

In [ ]:
### set directory
txt_fdiry = file.path(FD_DAT, "external", txt_folder)
txt_fname = "checksum_md5sum.txt"
txt_fpath = file.path(txt_fdiry, txt_fname)

### write table
dat = dat_download_md5sum
write_delim(dat, txt_fpath, delim = " ", col_names = FALSE)

In [ ]:
### set output path
txt_fdiry = file.path(FD_RES, "region", txt_folder, "summary")
txt_fname = "metadata.tsv"
txt_fpath = file.path(txt_fdiry, txt_fname)

### save table
dat = dat_metadata_histone
write_tsv(dat, txt_fpath)  